# Day 2 Hands-on Summary Exercise



In [ ]:

import os
import sys
import json
import time
import math
import pickle
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn
import mlflow.pyfunc
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.datasets import (
    load_iris, load_diabetes, load_digits, load_linnerud,
    load_wine, load_breast_cancer, make_classification, make_regression
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, ElasticNet
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor

warnings.filterwarnings("ignore")


def find_project_root(start_path=None):
    """Find the mlflow_for_ml_dev folder from wherever Jupyter is started."""
    start = Path(start_path or Path.cwd()).resolve()
    for parent in [start] + list(start.parents):
        if parent.name == "mlflow_for_ml_dev" and (parent / "notebooks").exists():
            return parent
        if (parent / "datasets").exists() and (parent / "mlruns").exists() and (parent / "notebooks").exists():
            return parent
    # Fallback: if user started Jupyter inside notebooks subfolder
    for parent in [start] + list(start.parents):
        if parent.name == "notebooks":
            return parent.parent
    return start

PROJECT_ROOT = find_project_root()
DATASETS_DIR = PROJECT_ROOT / "datasets"
MLRUNS_DIR = PROJECT_ROOT / "mlruns"
MODELS_DIR = PROJECT_ROOT / "models"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "artifacts_generated"

for folder in [DATASETS_DIR, MLRUNS_DIR, MODELS_DIR, ARTIFACTS_DIR, MLRUNS_DIR / ".trash"]:
    folder.mkdir(parents=True, exist_ok=True)

# IMPORTANT: Every notebook logs to the same mlruns folder inside this project.
mlflow.set_tracking_uri(MLRUNS_DIR.as_uri())
client = MlflowClient()

print("Project root      :", PROJECT_ROOT)
print("Datasets folder   :", DATASETS_DIR)
print("MLflow runs folder:", MLRUNS_DIR)
print("Models folder     :", MODELS_DIR)
print("Tracking URI      :", mlflow.get_tracking_uri())


## Goal

This is the final practice notebook for MLflow Basics.

Students should complete the steps independently:

1. Load dataset
2. Train one model
3. Log params
4. Log metrics
5. Log model
6. Open MLflow UI
7. Compare runs

In [ ]:
# ============================================================
# Common setup
# ============================================================

# pandas is used for tabular data analysis.
import pandas as pd

# numpy is used for numerical operations.
import numpy as np

# matplotlib is used for simple visualizations.
import matplotlib.pyplot as plt

# pickle is used when we want to save/load normal sklearn objects.
import pickle

# MLflow is used to track experiments, parameters, metrics, models, and artifacts.
import mlflow
import mlflow.sklearn

# sklearn utilities for data splitting and model evaluation.
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

# Common algorithms used in this training.
from sklearn.linear_model import LogisticRegression, LinearRegression, ElasticNet
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# We use local file-based MLflow tracking.
# This creates an mlruns folder inside the project folder.
# MLflow tracking URI is configured in the first setup cell.

print("Setup completed.")
print("MLflow tracking URI:", mlflow.get_tracking_uri())

In [ ]:
# ============================================================
# Read classification dataset
# ============================================================

# This dataset contains customer details and a target column called churn.
# churn = 1 means the customer may leave.
# churn = 0 means the customer may stay.
df = pd.read_csv("../../datasets/customer_churn_1000.csv")

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
# ============================================================
# Prepare classification data
# ============================================================

# Separate input features and target.
X = df.drop("churn", axis=1)
y = df["churn"]

# Convert categorical columns to numeric columns using one-hot encoding.
# This keeps preprocessing simple for students.
X = pd.get_dummies(X, drop_first=True)

# Split data into training and testing data.
# stratify=y keeps class distribution similar in train and test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

In [ ]:
# ============================================================
# Student exercise
# ============================================================

mlflow.set_experiment("student_day_summary_exercises")

# Change these values and run multiple times.
n_estimators = 80
max_depth = 5

with mlflow.start_run(run_name="student_random_forest_exercise"):

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mlflow.log_param("model_name", "RandomForestClassifier")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)

    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("precision", precision_score(y_test, y_pred))
    mlflow.log_metric("recall", recall_score(y_test, y_pred))
    mlflow.log_metric("f1_score", f1_score(y_test, y_pred))

    mlflow.sklearn.log_model(model, name="model")

    print("Exercise completed. Check MLflow UI.")